# IndieFake (Indian-accent) SSL training pipeline -- Colab

Picks up the `ai_voice_detector` pipeline at step 4 (SSL feature extraction), which was too slow on an 8GB CPU-only machine. Uses Colab's GPU (`Runtime > Change runtime type > T4 GPU`) instead.

**Before running**: put the 4 IndieFake zip parts (`drive-download-...-00{1..4}.zip`) somewhere in your Google Drive and set `INDIEFAKE_ZIP_DIR` below to that folder.

Steps: mount Drive -> clone repo -> re-fetch ASVspoof + real-world audio (public/deterministic sources, not stored in git) -> run the Indian dataset pipeline (organize -> verify -> degrade -> manifests -> extract embeddings on GPU) -> train -> evaluate (5-quadrant + leave-one-generator-out) -> push results back.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# EDIT THIS to wherever the 4 IndieFake zip parts live in your Drive
INDIEFAKE_ZIP_DIR = "/content/drive/MyDrive/IndieFake  Dataset"

In [ ]:
REPO_URL = "https://github.com/Jeevan-Cyber-Sai/sih.git"
!git clone $REPO_URL /content/sih
%cd /content/sih/ai_voice_detector

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- set Runtime > Change runtime type > T4 GPU')

## Re-fetch ASVspoof + real-world audio

`data/`, `data_realworld/` audio isn't in git (gitignored, large binary data) -- only their SSL embeddings are (`cache/ssl_embeddings/`, portable across machines since the cache key is now a relative path, not an absolute one). These scripts rebuild the audio deterministically (same seeds, same public sources) so file *names* line up with what's already cached; `LA_D_*`/WhatsApp/test-dir files come from the same seeded/tracked sources so they're byte-identical, while gTTS/LibriSpeech content could differ by a negligible amount from re-synthesis -- acceptable since only filenames need to match for cache lookups, and any recompute needed for a mismatch is cheap on GPU anyway.

In [ ]:
!python scripts/download_asvspoof_subset.py 700

In [ ]:
!python scripts/build_realworld_dataset.py

In [ ]:
!python scripts/build_realworld_fake_dataset.py

## Indian dataset pipeline (steps 1-3, replayed here since they're cheap)

In [ ]:
import os
os.environ['INDIEFAKE_ZIP_DIR'] = INDIEFAKE_ZIP_DIR
!python scripts/organize_indian_dataset.py

In [ ]:
!python verify_indian_dataset.py

In [ ]:
!python scripts/degrade_indian_dataset.py

In [ ]:
# Idempotent -- these will report 0 new entries since the manifests are
# already committed to git and cloned above. Harmless to (re-)run.
!python scripts/build_holdout_manifest_indian.py
!python scripts/build_generator_manifest_indian.py

## Step 4: extract SSL embeddings on GPU

`features_ssl.py` now auto-detects CUDA and moves the model/inputs there. `MAX_WORKERS=1` in the script is intentional even here -- one process already saturates a single GPU; more workers would just fight over it via separate CUDA contexts.

In [ ]:
!python scripts/extract_indian_ssl_features.py

## Step 5: train the combined classifier

In [ ]:
!python train_ssl.py --indian

## Step 6: six-bucket evaluation (clean / real-world / Indian x real / fake)

In [ ]:
!python scripts/five_quadrant_eval.py

## Step 7: leave-one-generator-out, including the new 'indiefake' generator

In [ ]:
!python scripts/leave_one_generator_out_eval.py

## Bring results back

`models/*.joblib` and `features_indian.npy` are gitignored (large/regenerable) -- copy them to Drive to download. The new Indian embeddings in `cache/ssl_embeddings/` ARE meant to be committed (same convention as the existing ASVspoof/real-world cache) so any machine that later `git pull`s gets them for free -- this needs a GitHub token with push access to this repo, entered interactively below (never hardcode it in the notebook).

In [ ]:
!mkdir -p /content/drive/MyDrive/sih_indian_results
!cp models/voice_classifier_ssl_indian.joblib models/scaler_ssl_indian.joblib features_indian.npy /content/drive/MyDrive/sih_indian_results/
print('copied to Drive: sih_indian_results/')

In [ ]:
import getpass
token = getpass.getpass('GitHub personal access token (repo scope, push rights): ')
!git config user.email "jeevanarhack@gmail.com"
!git config user.name "Jeevan Sai V"
!git add cache/ssl_embeddings data_realworld/holdout_manifest.json data_realworld/generator_manifest.json
!git commit -m "Add Indian-accent SSL embeddings from Colab run"
!git push https://{token}@github.com/Jeevan-Cyber-Sai/sih.git HEAD:main